taks 5

In [ ]:
!pip install -U datasets huggingface_hub fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: 

In [ ]:
!pip uninstall -y datasets -q
!pip install datasets==2.19.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 12.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


getting laod dataset from huggng face

In [ ]:
from datasets import load_dataset

dataset = load_dataset("empathetic_dialogues")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for empathetic_dialogues contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/empathetic_dialogues
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
 

Generating train split:   0%|          | 0/76673 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12030 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10943 [00:00<?, ? examples/s]

now laoding teh train file and cheking inside data

In [ ]:
train_data = dataset["train"]
print(train_data[0])

{'conv_id': 'hit:0_conv:1', 'utterance_idx': 1, 'context': 'sentimental', 'prompt': 'I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.', 'speaker_idx': 1, 'utterance': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.', 'selfeval': '5|5|5_2|2|5', 'tags': ''}


now creatng pairs because the ttransformer model learn supervisd mapping and using conversation turns as chatbot should learn contextual emotional responses

In [ ]:
def build_pairs(data):
    pairs = []
    conversation = []

    current_conv_id = None

    for row in data:
        conv_id = row["conv_id"]

        #conversation
        if current_conv_id != conv_id:
            conversation = []
            current_conv_id = conv_id

        conversation.append(row["utterance"])

        # creatng pairs when atlest 2 turn
        if len(conversation) >= 2:
            pairs.append({
                "input": conversation[-2],
                "response": conversation[-1]
            })

    return pairs

In [ ]:
train_pairs = build_pairs(train_data)

Converted raw conversation into sequential respons pairs for language modeling, as following is teh output:

In [ ]:
print(train_pairs[:5])

[{'input': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.', 'response': 'Was this a friend you were in love with_comma_ or just a best friend?'}, {'input': 'Was this a friend you were in love with_comma_ or just a best friend?', 'response': 'This was a best friend. I miss her.'}, {'input': 'This was a best friend. I miss her.', 'response': 'Where has she gone?'}, {'input': 'Where has she gone?', 'response': 'We no longer talk.'}, {'input': 'We no longer talk.', 'response': 'Oh was this something that happened because of an argument?'}]


Now tokenization, converting human readable text into a format that the ai model can understand.
Models like DistilGPT2 does not understand words directly they only understand numbers so we take each training example (“User” message and “Bot” response) and combine them into one single text string in a structured format like “User: Bot:” and then the tokenizer breaks this text into small pieces called tokens and convert them into numerical id.
I also add a special end token so the model know where a response finished So in this step prepares the data in a standardized numerical format so the model can learn patterns from conversation during fine tune

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def format_data(example):
    return {
        "text": f"User: {example['input']}\nBot: {example['response']}{tokenizer.eos_token}"
    }

In [ ]:
formatted_data = [format_data(x) for x in train_pairs]

Now converting data into dataset format because trainer api only works with hugging hace dataset format

In [ ]:
from datasets import Dataset

dataset_hf = Dataset.from_list(formatted_data)

tokenizing dataset as earlier we formatted text so now convertng it into model readable numbers

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
tokenized_dataset = dataset_hf.map(tokenize_function, batched=True)

Map:   0%|          | 0/58829 [00:00<?, ? examples/s]

Loading Model

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

configurng training

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=100,
    save_steps=500
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,1.310164
200,0.923252
300,0.935509
400,0.926761
500,0.928799
600,0.907823
700,0.942881
800,0.971976
900,0.911208
1000,0.907869


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14708, training_loss=0.8748807737870744, metrics={'train_runtime': 2464.4624, 'train_samples_per_second': 23.871, 'train_steps_per_second': 5.968, 'total_flos': 1921478317572096.0, 'train_loss': 0.8748807737870744, 'epoch': 1.0})